# Etapa 7 — Predictor de abandono

Análisis que respalda `src/ml_abandono.py` (decisiones **D58**, **D64**, **D65** y **D66** en `docs/decisiones.md`).

**Pregunta:** de los alumnos que vienen entrando al aula, ¿quiénes van a dejar de entrar en las próximas 2 semanas?
Sirve para que el tutor contacte a tiempo a quien está por dejar, en vez de enterarse cuando ya lleva semanas sin entrar.

**Objetivo (definición operativa).** A una fecha de corte *c*, un alumno de la base que tuvo **alguna interacción en los
últimos 14 días** *abandona* si **no tiene ninguna interacción en los 14 días siguientes** (c, c + 14].

**Variables:** todo lo que se sabía a la fecha de corte, sin nada del futuro: actividad reciente, ritmo propio del alumno,
lo que hace en sus otras materias, días desde la última evaluación de la materia y **notas a la fecha**. Cada nota cuenta
desde el día en que ese alumno rindió (su evento "Realizar evaluación"), no la foto de notas del 24/10.

**Historia del modelo.**
* **D58 (versión 1):** 11 variables, solo actividad; sin notas. Elegida: regresión logística.
* **D64 (versión 2):** un compañero encontró alumnos que **promocionan** marcados con riesgo alto. Se sumaron 15
  variables (ritmo, calendario, notas fechadas, otras materias, mezcla) y el gradient boosting.
* **D65:** gradient boosting poco profundo con **restricciones monótonas** (el riesgo nunca baja por llevar más días sin entrar).
* **D66:** las notas también con sentido obligado: una nota mejor nunca sube el riesgo.

El notebook solo **analiza**: todo el cálculo está en `src/ml_abandono.py`, que es lo que usa la app.
Para correrlo hace falta `pip install jupyter` (no está en `requirements.txt` porque la app no lo necesita).
Tarda unos 3 minutos (la sección 6 entrena 30 veces cada modelo).

In [1]:
import dataclasses
import os
import warnings
from pathlib import Path
if not Path("src").exists():          # el notebook está en notebooks/: subo a la raíz
    os.chdir("..")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from sklearn.base import clone
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold

from src import metricas as m
from src import ml_abandono as ab
from src.config import ABANDONO_HORIZONTE_DIAS as H, ABANDONO_SEMILLA

warnings.simplefilter("ignore")
pio.renderers.default = "notebook_connected"
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 90)
d = m.cargar()

## 1. Cómo se ve la cursada

Alumnos de la base que entraron cada semana. La cursada va del 05/08 al 24/10/2025 (12 semanas). La semana del 01/09 cae
en las tres materias (¿receso o semana sin clases?): es justamente donde más "abandonos" de 2 semanas aparecen.

In [2]:
ev = d.interacciones[d.interacciones["en_base"]]
semana = ev["fecha"] - pd.to_timedelta(ev["fecha"].dt.day_of_week, unit="D")
activos = ev.assign(semana=semana).groupby(["semana", "asignatura"])["id_alumno"].nunique().rename("alumnos").reset_index()
fig = px.line(activos, x="semana", y="alumnos", color="asignatura", markers=True,
              title="Alumnos de la base que entraron cada semana (líneas punteadas: evaluaciones)")
for f in d.instrumentos["fecha_instrumento"].dropna().unique():
    fig.add_vline(x=f, line_dash="dot", line_color="lightgray")
fig

## 2. Las fotos: de dónde salen los casos

No hay una fila "abandonó / no abandonó" en los datos: hay que construirla. Se toman **fotos** de la cursada cada 3 días
(no cada 7, para que caigan en distintos días de la semana: cada materia tiene su día fuerte) desde el día 14 hasta 14
días antes del último dato, para poder saber qué pasó después. En cada foto entran los alumnos que vienen entrando.

In [3]:
ds = ab.dataset(d)
por_foto = ds.groupby("fecha_corte").agg(alumnos=("abandona", "size"), abandonan=("abandona", "sum"),
                                         tasa=("abandona", "mean"))
por_foto["tasa"] = (100 * por_foto["tasa"]).round(1)
print(f"{len(ds)} casos, {ds['id_alumno'].nunique()} alumnos distintos, {len(por_foto)} fotos "
      f"({por_foto.index.min():%d/%m} a {por_foto.index.max():%d/%m}); abandona el {100 * ds['abandona'].mean():.1f} %")
por_foto

9678 casos, 226 alumnos distintos, 18 fotos (19/08 a 09/10); abandona el 10.7 %


,alumnos,abandonan,tasa
fecha_corte,,,
2025-08-19,594,57,9.6
2025-08-22,582,77,13.2
2025-08-25,572,83,14.5
2025-08-28,573,94,16.4
2025-08-31,574,91,15.9
2025-09-03,552,69,12.5
2025-09-06,526,40,7.6
2025-09-09,495,37,7.5
2025-09-12,501,33,6.6


### ¿Los que "abandonan" vuelven?

"Dejar de entrar 14 días" no es la baja formal. Para las fotos que tienen al menos 4 semanas por delante, se mira si los
que abandonaron volvieron a entrar **después** de esas 2 semanas.

In [4]:
fin = ab.fin_datos(d)
vuelven = []
for corte, grupo in ds[ds["abandona"]].groupby("fecha_corte"):
    if corte + pd.Timedelta(days=2 * H) > fin:
        continue
    despues = ev[ev["fecha"] > corte + pd.Timedelta(days=H)]
    volvio = pd.MultiIndex.from_frame(grupo[ab.CLAVES]).isin(pd.MultiIndex.from_frame(despues[ab.CLAVES]))
    vuelven.append({"fecha_corte": corte, "abandonan": len(grupo), "vuelven_despues": int(volvio.sum())})
vuelven = pd.DataFrame(vuelven)
print(f"Volvieron a entrar más adelante: {vuelven['vuelven_despues'].sum()} de {vuelven['abandonan'].sum()} "
      f"({100 * vuelven['vuelven_despues'].sum() / vuelven['abandonan'].sum():.0f} %)")
vuelven

Volvieron a entrar más adelante: 435 de 745 (58 %)


,fecha_corte,abandonan,vuelven_despues
0,2025-08-19,57,36
1,2025-08-22,77,52
2,2025-08-25,83,64
3,2025-08-28,94,72
4,2025-08-31,91,68
5,2025-09-03,69,48
6,2025-09-06,40,22
7,2025-09-09,37,17
8,2025-09-12,33,10
9,2025-09-15,27,1


## 3. Las notas, sin mirar el futuro (D64)

La versión 1 no usaba notas porque el Excel de calificaciones es una foto al 24/10: usarlo en una foto del 01/09 le
contaría al modelo notas que todavía no existían. Pero la trazabilidad tiene un evento **"Realizar evaluación"** por
alumno, y su descripción nombra al instrumento. Con eso cada nota se fecha **el día en que ese alumno rindió**:

* una evaluación está "tomada" si su fecha (mediana de las realizaciones, D18) ya pasó o si el alumno ya la rindió;
* lo tomado y no rendido vale 0 (D16); el recuperatorio reemplaza desde el día en que se rinde (D19);
* los instrumentos sin ningún evento en la trazabilidad (PyE "Evaluación Semanal Nº3") no se usan.

In [5]:
real = ab._realizaciones(d)
print(f"{len(real)} realizaciones ligadas a {real['id_instrumento'].nunique()} instrumentos")
alumno = real[real["id_instrumento"].eq("ALG-03")].sort_values("rindio_el").iloc[-1]
clave = (alumno["asignatura"], alumno["id_alumno"])
dia = alumno["rindio_el"].normalize()
cols = ["nota_pct", "margen_promocion", "pct_rendidos", "rindio_ultima"]
pd.DataFrame({f"{c:%d/%m}": ab.variables(d, c).set_index(ab.CLAVES).loc[clave, cols]
              for c in [dia - pd.Timedelta(days=1), dia]}).rename_axis(f"{clave[1]}, rindió el 1° Parcial el {dia:%d/%m}")

1782 realizaciones ligadas a 17 instrumentos


,10/10,11/10
"ALU_1143, rindió el 1° Parcial el 11/10",,
nota_pct,100.00,82.22
margen_promocion,29.17,11.39
pct_rendidos,100.00,100.00
rindio_ultima,1.00,1.00


**Prueba de que no hay fuga:** se calculan las variables con todos los datos y con los datos cortados en la fecha de la
foto (borrando toda interacción posterior). Si alguna variable usara el futuro, darían distinto. (Lo mismo verifica
`tests/test_ml_abandono.py`.)

In [6]:
for corte in ["2025-08-25", "2025-09-20", "2025-10-05"]:
    fin_dia = pd.Timestamp(corte) + pd.Timedelta(days=1)
    cortado = dataclasses.replace(d, interacciones=d.interacciones[d.interacciones["fecha_hora"] < fin_dia],
                                  sesiones=d.sesiones[d.sesiones["inicio"] < fin_dia])
    a = ab.variables(d, corte).set_index(ab.CLAVES)[list(ab.VARIABLES)]
    b = ab.variables(cortado, corte).set_index(ab.CLAVES).loc[a.index, list(ab.VARIABLES)]
    print(corte, "iguales:", a.equals(b))

2025-08-25 iguales: True


2025-09-20 iguales: True


2025-10-05 iguales: True


## 4. Variables: en qué se diferencian los que abandonan

Las 26 variables, por familia, con su mediana según el desenlace.

In [7]:
familias = {**{v: "1. actividad" for v in list(ab.VARIABLES)[:11]},
            **{v: "2. ritmo propio" for v in ["dias_activos_total", "pausa_habitual", "sin_vs_habitual"]},
            **{v: "3. calendario" for v in ["dias_desde_evaluacion", "entro_despues_evaluacion"]},
            **{v: "4. notas a la fecha" for v in ["tiene_notas", "nota_pct", "margen_promocion", "pct_rendidos", "rindio_ultima"]},
            **{v: "5. otras materias" for v in ["eventos_14d_otras", "materias_activas_14d", "dias_sin_actividad_global"]},
            **{v: "6. mezcla" for v in ["pct_evaluacion", "pct_notas"]}}
medianas = ds.groupby("abandona")[list(ab.VARIABLES)].median().T
medianas.columns = ["sigue", "abandona"]
medianas.insert(0, "descripcion", medianas.index.map(ab.VARIABLES))
medianas.insert(0, "familia", medianas.index.map(familias))
medianas

,familia,descripcion,sigue,abandona
dias_sin_actividad,1. actividad,Días desde la última vez que entró,2.000,5.0000
eventos_7d,1. actividad,Interacciones en los últimos 7 días,15.000,3.0000
eventos_14d,1. actividad,Interacciones en los últimos 14 días,37.000,17.0000
dias_activos_14d,1. actividad,Días con actividad en los últimos 14 días,4.000,2.0000
sesiones_14d,1. actividad,Sesiones en los últimos 14 días,5.000,2.0000
minutos_14d,1. actividad,Minutos en la plataforma en los últimos 14 días,29.400,10.3000
evaluaciones_14d,1. actividad,Interacciones de evaluación en los últimos 14 días,3.000,1.0000
tendencia,1. actividad,Tendencia: últimas 2 semanas contra las 2 anteriores (log),0.026,0.1485
pct_semanas_activas,1. actividad,% de las semanas de cursada en que entró,88.900,66.7000
eventos_total,1. actividad,Interacciones desde el inicio de la cursada,138.000,74.0000


In [8]:
tasa_dias = ds.groupby("dias_sin_actividad")["abandona"].agg(["size", "mean"]).rename(columns={"size": "casos", "mean": "tasa"})
tasa_dias["tasa"] = (100 * tasa_dias["tasa"]).round(1)
tasa_dias.T

dias_sin_actividad,0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0,13.0
casos,2730.0,1711.0,1185.0,853.0,743.0,563.0,408.0,339.0,270.0,222.0,190.0,169.0,159.0,136.0
tasa,3.9,6.7,7.7,9.4,11.7,13.5,17.4,20.1,25.2,27.5,26.3,34.9,32.7,35.3


## 5. ¿Los que promocionan abandonan?

Es la duda que originó la versión 2: "es raro que alguien que promociona abandone". Los datos dicen que es **más raro,
pero no raro**: el objetivo es dejar de entrar 2 semanas, y un alumno al día puede hacer una pausa larga (por ejemplo,
después de un parcial). Se agrupan los casos por su nota a la fecha contra el corte de promoción.

In [9]:
ds["nota_a_la_fecha"] = pd.cut(ds["margen_promocion"].where(ds["tiene_notas"] > 0), [-101, -40, -20, 0, 101],
                               labels=["muy lejos de promocionar (< -40)", "lejos (-40 a -20)", "cerca (-20 a 0)",
                                       "promociona"])
def tasa(g):
    return g["abandona"].agg(casos="size", abandona_pct=lambda s: round(100 * s.mean(), 1))
pd.concat({"todos": tasa(ds.groupby("nota_a_la_fecha", observed=True)),
           "con 7 días o más sin entrar": tasa(ds[ds["dias_sin_actividad"] >= 7].groupby("nota_a_la_fecha", observed=True))},
          axis=1)

todos              con 7 días o más sin entrar             
                                 casos abandona_pct                       casos abandona_pct
nota_a_la_fecha                                                                             
muy lejos de promocionar (< -40)  3371         17.6                         759         32.3
lejos (-40 a -20)                  953         12.7                         162         33.3
cerca (-20 a 0)                    854          7.3                          85         30.6
promociona                        4340          5.7                         465         16.3

## 6. Comparación de modelos

`ab.entrenar(d)` hace todo:

1. Separa el **25 % de los alumnos** para prueba (todas sus fotos y materias): `GroupShuffleSplit` por `id_alumno`.
   Si un alumno estuviera en las dos partes, el modelo lo "reconocería" y la prueba sería optimista.
2. En el 75 % restante, **validación cruzada por alumno** (`GroupKFold`, 5 partes): con esas predicciones elige el
   umbral de menor costo para cada modelo y el modelo de menor costo.
3. Mide en prueba la regresión logística, el Random Forest, el gradient boosting y la regla "más de 7 días sin entrar".
4. Reentrena el elegido con todos los datos.

In [10]:
mod = ab.entrenar(d)
print(f"Elegido: {mod.nombre}, umbral {mod.umbral:.2f}")
mod.comparacion.drop(columns=["elegido"])

Elegido: Gradient boosting, umbral 0.13


,modelo,conjunto,filas,abandonos,auc_roc,auc_pr,brier,precision,recall,pct_marcados,costo,costo_por_100
0,Regresión logística,validación cruzada,7367,766,0.800,0.304,0.0835,27.4,62.9,23.8,2694.0,36.6
1,Regresión logística,prueba,2311,266,0.829,0.384,0.0851,30.3,68.0,25.9,842.0,36.4
2,Random Forest,validación cruzada,7367,766,0.821,0.332,0.0796,28.7,65.8,23.8,2563.0,34.8
3,Random Forest,prueba,2311,266,0.829,0.364,0.0849,31.2,70.3,25.9,807.0,34.9
4,Gradient boosting,validación cruzada,7367,766,0.828,0.351,0.0789,28.2,69.7,25.7,2517.0,34.2
5,Gradient boosting,prueba,2311,266,0.814,0.348,0.0867,29.6,72.6,28.2,824.0,35.7
6,Regla simple,validación cruzada,7367,766,0.712,0.215,NaN,28.0,32.1,11.9,3233.0,43.9
7,Regla simple,prueba,2311,266,0.714,0.278,NaN,34.5,34.6,11.6,1045.0,45.2


### Una sola partición no alcanza

La prueba son unos 56 alumnos, y con tan pocos el orden de los modelos puede cambiar por azar. Para decidir con más
confianza se repite la validación cruzada por alumno con **5 particiones distintas** (cada alumno queda afuera una vez en
cada una) y se comparan los mismos modelos, más dos variantes para medir el aporte de las notas.

In [11]:
V = list(ab.VARIABLES)
NOTAS = ["tiene_notas", "nota_pct", "margen_promocion", "pct_rendidos", "rindio_ultima"]
X, y, g = ds[V].astype(float), ds["abandona"].to_numpy(), ds["id_alumno"].to_numpy()
promociona = ((ds["margen_promocion"] >= 0) & (ds["tiene_notas"] > 0)).to_numpy()

def sin_notas():
    est = clone(ab.modelos()["gradient_boosting"])
    cols = [c for c in V if c not in NOTAS]
    return est.set_params(sel__transformers=[("todas", "passthrough", cols)],
                          modelo__monotonic_cst=[ab.MONOTONAS.get(c, 0) for c in cols])

candidatos = {**{ab.NOMBRES_MODELOS[k]: e for k, e in ab.modelos().items()},
              "Gradient boosting sin notas": sin_notas()}
filas = []
for semilla in range(5):
    particion = list(StratifiedGroupKFold(5, shuffle=True, random_state=100 + semilla).split(X, y, g))
    for nombre, est in candidatos.items():
        prob = np.zeros(len(y))
        for tr, te in particion:
            prob[te] = clone(est).fit(X.iloc[tr], y[tr]).predict_proba(X.iloc[te])[:, 1]
        u = ab.elegir_umbral(y, prob); marcado = prob >= u
        filas.append({"modelo": nombre, "semilla": semilla, "auc_roc": roc_auc_score(y, prob),
                      "auc_pr": average_precision_score(y, prob), "costo_por_100": 100 * ab.costo(y, marcado) / len(y),
                      "promocionan_marcados": (marcado & promociona).sum(),
                      "acierto_en_ellos": 100 * y[marcado & promociona].mean()})
    filas.append({"modelo": ab.REGLA, "semilla": semilla, "auc_roc": roc_auc_score(y, X["dias_sin_actividad"]),
                  "auc_pr": average_precision_score(y, X["dias_sin_actividad"]),
                  "costo_por_100": 100 * ab.costo(y, X["dias_sin_actividad"] > ab.ABANDONO_REGLA_DIAS) / len(y)})
robusta = pd.DataFrame(filas)
robusta.groupby("modelo").agg(["mean", "std"]).drop(columns="semilla").round(3).sort_values(("costo_por_100", "mean"))

auc_roc        auc_pr        costo_por_100        promocionan_marcados         acierto_en_ellos       
                               mean    std   mean    std          mean    std                 mean     std             mean    std
modelo                                                                                                                            
Gradient boosting             0.828  0.003  0.356  0.008        33.928  0.263                480.4  73.660           27.213  1.846
Random Forest                 0.826  0.001  0.342  0.007        34.210  0.318                476.2  47.846           26.991  0.682
Gradient boosting sin notas   0.824  0.004  0.353  0.007        34.375  0.517                586.4  87.091           24.203  1.856
Regresión logística           0.811  0.003  0.324  0.009        35.817  0.323                427.4  53.575           26.562  1.126
Regla simple                  0.713  0.000  0.228  0.000        44.203  0.000                  NaN     NaN              NaN    NaN

## 7. Validación temporal

La validación por alumno mezcla semanas: el modelo ve fotos de octubre al entrenar. En la práctica se va a usar para
predecir **semanas que todavía no pasaron**, así que también se prueba entrenando con las fotos hasta el 20/09 y midiendo
en las posteriores (con alumnos de prueba distintos, para no mezclar las dos cosas).

In [12]:
e, p = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=ABANDONO_SEMILLA).split(X, y, g))
temprano = (ds["fecha_corte"] <= "2025-09-20").to_numpy()
ent = np.zeros(len(ds), bool); ent[e] = True
epocas = [("misma época (hasta 20/09)", ~ent & temprano), ("después (desde 21/09)", ~ent & ~temprano)]
filas = []
for clave, est in ab.modelos().items():
    ajuste = clone(est).fit(X[ent & temprano], y[ent & temprano])
    for nombre, mascara in epocas:
        prob = ajuste.predict_proba(X[mascara])[:, 1]
        filas.append({"modelo": ab.NOMBRES_MODELOS[clave], "prueba": nombre,
                      "auc_roc": roc_auc_score(y[mascara], prob), "auc_pr": average_precision_score(y[mascara], prob)})
for nombre, mascara in epocas:
    dias = X.loc[mascara, "dias_sin_actividad"]
    filas.append({"modelo": ab.REGLA, "prueba": nombre, "auc_roc": roc_auc_score(y[mascara], dias),
                  "auc_pr": average_precision_score(y[mascara], dias)})
pd.DataFrame(filas).round(3).pivot(index="modelo", columns="prueba", values=["auc_roc", "auc_pr"])

auc_roc                                          auc_pr                          
prueba              después (desde 21/09) misma época (hasta 20/09) después (desde 21/09) misma época (hasta 20/09)
modelo                                                                                                             
Gradient boosting                   0.816                     0.820                 0.378                     0.395
Random Forest                       0.842                     0.824                 0.389                     0.373
Regla simple                        0.756                     0.697                 0.283                     0.281
Regresión logística                 0.808                     0.832                 0.347                     0.412

## 8. Calibración: ¿la probabilidad se puede leer como probabilidad?

En los alumnos de prueba, se agrupan los casos en 10 partes según la probabilidad predicha y se compara con el % que
abandonó de verdad. Si están cerca, "30 %" quiere decir "de cada 10 así, abandonan 3".

In [13]:
ajuste = clone(ab.modelos()[mod.clave]).fit(X.iloc[e], y[e])
prob = ajuste.predict_proba(X.iloc[p])[:, 1]
cal = pd.DataFrame({"prob": prob, "real": y[p]})
cal["decil"] = pd.qcut(cal["prob"].rank(method="first"), 10, labels=False) + 1
cal = cal.groupby("decil").agg(predicha=("prob", "mean"), real=("real", "mean"), casos=("real", "size"))
cal[["predicha", "real"]] = (100 * cal[["predicha", "real"]]).round(1)
fig = px.scatter(cal, x="predicha", y="real", size="casos", title=f"Calibración en los alumnos de prueba (%), {mod.nombre}")
fig.add_shape(type="line", x0=0, y0=0, x1=60, y1=60, line=dict(dash="dash"))
fig.show()
cal

,predicha,real,casos
decil,,,
1,0.8,0.0,232
2,1.5,1.7,231
3,2.2,3.9,231
4,3.1,2.2,231
5,4.5,4.8,231
6,6.3,9.1,231
7,9.4,7.8,231
8,15.0,17.7,231
9,23.6,26.8,231


## 9. El umbral según el costo

El umbral sale de cuánto cuesta cada error (como en la clase 10). Con costo 5 a 1 (no detectar a quien abandona cuesta
5 veces más que contactar de más) el umbral es bajo. La tabla muestra qué pasaría con otras proporciones: es la
discusión que hay que tener con la coordinación, no una decisión técnica.

In [14]:
filas = []
for fn in [1, 2, 3, 5, 10]:
    u = ab.elegir_umbral(y[p], prob, costo_fn=fn, costo_fp=1)
    t = ab.tabla_umbral(y[p], prob, costo_fn=fn).set_index("umbral").loc[u]
    filas.append({"costo_fn : costo_fp": f"{fn} : 1", "umbral": u, "% marcados": t["pct_marcados"],
                  "precisión": t["precision"], "recall": t["recall"]})
pd.DataFrame(filas)

,costo_fn : costo_fp,umbral,% marcados,precisión,recall
0,1 : 1,0.54,1.25,58.62,6.39
1,2 : 1,0.35,6.62,46.41,26.69
2,3 : 1,0.21,17.14,35.86,53.38
3,5 : 1,0.13,28.21,29.60,72.56
4,10 : 1,0.11,31.41,27.82,75.94


In [15]:
curva = mod.curva.assign(umbral=lambda c: 100 * c["umbral"])
fig = px.line(curva, x="umbral", y="costo", title="Costo total según el umbral (alumnos de prueba, costo 5 : 1)")
fig.add_vline(x=100 * mod.umbral, line_dash="dash")
fig

## 10. Qué variables usa el modelo

Importancia por permutación en los alumnos de prueba: cuánto empeora el AUC PR si se mezclan al azar los valores de
una variable. Las variables están muy correlacionadas entre sí (interacciones de 7 y de 14 días, sesiones, minutos; días
sin entrar en la materia y en todas), así que cuando una "no importa" suele ser porque otra ya dice lo mismo. Por eso
también se suma por familia.

In [16]:
imp = mod.importancia.assign(familia=lambda t: t["variable"].map(familias))
display(imp.groupby("familia")["importancia"].sum().sort_values(ascending=False).round(4).to_frame("importancia total"))
imp

,importancia total
familia,
1. actividad,0.0707
5. otras materias,0.0403
2. ritmo propio,0.0118
3. calendario,0.0091
4. notas a la fecha,-0.0092
6. mezcla,-0.0120


,variable,descripcion,importancia,desvio,familia
0,dias_sin_actividad_global,Días desde que entró a cualquiera de sus materias,0.0234,0.0024,5. otras materias
1,eventos_7d,Interacciones en los últimos 7 días,0.0176,0.0041,1. actividad
2,dias_de_cursada,Días desde el inicio de la cursada,0.0146,0.0064,1. actividad
3,eventos_14d_otras,Interacciones en sus otras materias en los últimos 14 días,0.0142,0.0027,5. otras materias
4,pct_semanas_activas,% de las semanas de cursada en que entró,0.0130,0.0057,1. actividad
5,tendencia,Tendencia: últimas 2 semanas contra las 2 anteriores (log),0.0115,0.0028,1. actividad
6,dias_activos_14d,Días con actividad en los últimos 14 días,0.0108,0.0029,1. actividad
7,pausa_habitual,"Su pausa habitual entre días activos (percentil 90, días)",0.0097,0.0035,2. ritmo propio
8,dias_desde_evaluacion,Días desde la última evaluación de la materia,0.0091,0.0068,3. calendario
9,sesiones_14d,Sesiones en los últimos 14 días,0.0064,0.0019,1. actividad


### Restricciones monótonas (D65 y D66)

El gradient boosting tiene el **sentido obligado** en algunas variables: más días sin entrar nunca bajan el riesgo, y una
nota mejor, más evaluaciones rendidas o haber rendido la última nunca lo suben. No cambia el rendimiento (sección 6),
pero evita que el modelo aprenda patrones absurdos de una sola cursada y es más fácil de defender. El gráfico muestra
el riesgo promedio de los alumnos de las fotos si a todos se les cambiara solo esa variable (dependencia parcial).

In [17]:
from sklearn.inspection import partial_dependence
graficos = []
for var in ["margen_promocion", "dias_sin_actividad", "eventos_14d_otras"]:
    pdp = partial_dependence(mod.modelo, X, [var], grid_resolution=30, kind="average")
    graficos.append(pd.DataFrame({"valor": pdp["grid_values"][0], "riesgo_promedio": 100 * pdp["average"][0],
                                  "variable": ab.VARIABLES[var]}))
px.line(pd.concat(graficos), x="valor", y="riesgo_promedio", facet_col="variable", facet_col_spacing=0.06,
        title="Riesgo promedio (%) al cambiar una sola variable").update_xaxes(matches=None) \
  .for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

## 11. A la fecha de corte por defecto (24/10/2025)

In [18]:
mod_app = ab.cargar_modelo(d)       # el mismo que usa la app (lo reentrena si cambiaron los datos)
riesgo = ab.riesgo_abandono(d, modelo=mod_app)
aviso = riesgo.attrs["aviso"]
riesgo = riesgo.merge(m.nota_final(d)[ab.CLAVES + ["condicion"]], on=ab.CLAVES)
print(aviso, end="\n\n")
pd.crosstab(riesgo["condicion"], riesgo["nivel_riesgo"], margins=True)

30 de los alumnos en riesgo alto (7 de Algoritmos y Estructuras de Datos, 4 de Probabilidad y Estadística, 19 de Álgebra) entraron por última vez a rendir una evaluación posterior al 09/10, la última foto con desenlace conocido. Después de una evaluación es normal una pausa y el modelo todavía no vio cómo siguen esas pausas, así que en ellos puede exagerar el riesgo.



nivel_riesgo,Alto,Bajo,Medio,Sin actividad,Ya abandonó,All
condicion,,,,,,
Libre,107,15,27,7,169,325
Promociona,29,98,27,0,2,156
Regular,45,72,30,0,9,156
All,181,185,84,7,180,637


Los que promocionan y quedan en riesgo alto, con lo que dice la página de cada uno. La columna `advertencia` aparece
cuando su último ingreso fue para rendir una evaluación posterior a la última foto con desenlace: el modelo todavía no
vio cómo sigue esa pausa.

In [19]:
alto_promo = riesgo[riesgo["nivel_riesgo"].eq(ab.ALTO) & riesgo["condicion"].eq("Promociona")]
print(f"{len(alto_promo)} alumnos; {alto_promo['advertencia'].ne('').sum()} con advertencia")
alto_promo[["asignatura", "id_alumno", "probabilidad", "dias_sin_actividad", "motivos", "advertencia"]]

29 alumnos; 13 con advertencia


,asignatura,id_alumno,probabilidad,dias_sin_actividad,motivos,advertencia
422,Álgebra,ALU_1157,57.6,13.0,13 días sin entrar; entró solo 1 día(s) en las últimas 2 semanas; tampoco entra a sus ...,Su último ingreso fue para rendir «1° Parcial» el 11/10. Después de una evaluación es ...
427,Álgebra,ALU_1158,50.6,13.0,13 días sin entrar; su actividad bajó de 17 a 5 interacciones (últimas 2 semanas contr...,Su último ingreso fue para rendir «1° Parcial» el 11/10. Después de una evaluación es ...
428,Álgebra,ALU_1248,47.6,13.0,13 días sin entrar; entró solo 1 día(s) en las últimas 2 semanas; lleva más días sin e...,Su último ingreso fue para rendir «1° Parcial» el 11/10. Después de una evaluación es ...
435,Álgebra,ALU_1150,36.4,12.0,12 días sin entrar; entró solo 2 día(s) en las últimas 2 semanas; lleva más días sin e...,Su último ingreso fue para rendir «1° Parcial» el 11/10. Después de una evaluación es ...
438,Álgebra,ALU_1301,35.1,10.0,10 días sin entrar; entró solo 2 día(s) en las últimas 2 semanas; lleva más días sin e...,
441,Álgebra,ALU_1035,34.1,12.0,12 días sin entrar; su actividad bajó de 16 a 7 interacciones (últimas 2 semanas contr...,
443,Álgebra,ALU_1263,33.8,13.0,13 días sin entrar; entró solo 1 día(s) en las últimas 2 semanas; entró en el 42 % de ...,Su último ingreso fue para rendir «1° Parcial» el 11/10. Después de una evaluación es ...
445,Álgebra,ALU_1092,33.2,12.0,12 días sin entrar; entró solo 2 día(s) en las últimas 2 semanas; tampoco entra a sus ...,Su último ingreso fue para rendir «1° Parcial» el 11/10. Después de una evaluación es ...
446,Álgebra,ALU_1138,31.9,11.0,11 días sin entrar; entró solo 2 día(s) en las últimas 2 semanas; lleva más días sin e...,
447,Álgebra,ALU_1128,31.3,5.0,Su actividad bajó de 20 a 2 interacciones (últimas 2 semanas contra las 2 anteriores);...,


### ¿Y en fechas que ya pasaron?

Ojo: estas fechas están dentro del entrenamiento, así que las cifras son optimistas. Sirven para ver la tendencia, no
para medir el modelo (eso es la sección 6).

In [20]:
filas = []
for corte in ["2025-09-15", "2025-09-20", "2025-10-01", "2025-10-08"]:
    r = ab.riesgo_abandono(d, corte, modelo=mod_app).merge(m.nota_final(d, corte)[ab.CLAVES + ["condicion"]], on=ab.CLAVES)
    alto = r[r["nivel_riesgo"].eq(ab.ALTO)]
    promo = alto[alto["condicion"].eq("Promociona")]
    filas.append({"fecha de corte": corte, "riesgo alto": len(alto),
                  "% que dejó de entrar": round(100 * alto["abandono_real"].astype(float).mean(), 1),
                  "promocionan en riesgo alto": len(promo),
                  "% de ellos que dejó de entrar": round(100 * promo["abandono_real"].astype(float).mean(), 1)})
pd.DataFrame(filas)

,fecha de corte,riesgo alto,% que dejó de entrar,promocionan en riesgo alto,% de ellos que dejó de entrar
0,2025-09-15,90,22.2,23,17.4
1,2025-09-20,117,29.1,20,15.0
2,2025-10-01,107,34.6,16,12.5
3,2025-10-08,157,35.0,14,14.3


## Conclusiones

* **Casos:** 18 fotos (19/08 a 09/10) dan 9.678 casos de 226 alumnos distintos; abandona el **10,7 %** (de 5 % a
  16 % según la semana; el pico es alrededor de la semana del 01/09, que tiene poca actividad en las tres materias).
* **Abandonar 14 días no es irse:** más de la mitad de los que dejaron de entrar 2 semanas volvió más adelante (sección 2).
  Por eso el resultado se usa para **priorizar contactos**, no para dar a nadie por perdido.
* **Las notas se pueden usar sin mirar el futuro:** cada nota se fecha con el evento "Realizar evaluación" del alumno, y
  las variables dan exactamente igual si se borran los datos posteriores a la foto (sección 3).
* **¿Es raro que alguien que promociona abandone?** Es **menos frecuente, pero pasa**: abandona el 5,7 % de los casos
  que promocionan a la fecha, contra 17,6 % de los que están muy lejos de promocionar. Y entre los que ya llevan 7 días
  o más sin entrar, el 16 % de los que promocionan deja de entrar 2 semanas (sección 5). Como el umbral de "riesgo alto"
  es bajo (13 %, por el costo 5 : 1), un alumno que promociona pero lleva casi 2 semanas sin entrar puede quedar arriba.
  "Riesgo alto" quiere decir "vale la pena un mensaje", no "lo más probable es que abandone".
* **Las notas ayudan, aunque no sean lo que más pesa.** Sin ellas el costo sube de 33,9 a 34,4 cada 100 casos y se marca
  en riesgo alto a un 22 % más de alumnos que promocionan (586 contra 480 en las fotos), con menos acierto en ellos
  (24 % contra 27 %). En la importancia por permutación casi no aparecen porque se superponen con la actividad: quien
  tiene buena nota suele entrar mucho (sección 10).
* **Modelo elegido: gradient boosting poco profundo con restricciones monótonas** (D65 y D66). Con 5 particiones
  distintas por alumno: costo 33,9 contra 34,2 del Random Forest, 35,8 de la logística y 44,2 de la regla de los 7 días;
  AUC ROC 0,83 contra 0,71 de la regla. En la partición de prueba fija sale un poco peor que el Random Forest (35,7
  contra 34,9), pero es una sola partición de 56 alumnos.
* **Probabilidades confiables:** los deciles caen cerca de la diagonal (sección 8): "30 %" se puede leer como "3 de cada 10".
* **Qué mira:** sobre todo la actividad reciente y lo que hace en sus **otras materias** (dejar de entrar a todas es la
  señal más fuerte), después su ritmo propio y los días desde la última evaluación.
* **Umbral:** con costo 5 : 1 es 13 % y se marca a 1 de cada 4 alumnos que vienen entrando. Es una decisión de la
  coordinación: con 3 : 1 sube a 21 % y se marca a 1 de cada 6; con 2 : 1, a 35 % y se marca a 1 de cada 15 (pero se
  detecta solo a 1 de cada 4 que abandonan).
* **Al 24/10** quedan 29 alumnos que promocionan en riesgo alto, todos de Álgebra, la mayoría con 10 a 13 días sin
  entrar desde el 1° Parcial (11/10). Las fotos con desenlace llegan al 09/10, así que el modelo no vio ninguna pausa después de ese
  parcial; a los que entraron por última vez a rendir, la app les muestra una advertencia.
* **Llegó a su techo con estos datos:** también se probaron otros parámetros, ensambles, fotos diarias, pesos por alumno
  y el calendario de las evaluaciones futuras, y ninguno mejora más que el ruido (D65). Para mejorar hace falta **otra
  cursada para entrenar**, no otro modelo.
* **Limitaciones:** una sola cursada y 226 personas; los casos de un mismo alumno no son independientes (por eso la
  validación es por alumno); el objetivo es "no entrar al aula", no la baja formal.